# Trexquant Interview Project (The Hangman Game)

* Copyright Trexquant Investment LP. All Rights Reserved. 
* Redistribution of this question without written consent from Trexquant is prohibited

## Instruction:
For this coding test, your mission is to write an algorithm that plays the game of Hangman through our API server. 

When a user plays Hangman, the server first selects a secret word at random from a list. The server then returns a row of underscores (space separated)—one for each letter in the secret word—and asks the user to guess a letter. If the user guesses a letter that is in the word, the word is redisplayed with all instances of that letter shown in the correct positions, along with any letters correctly guessed on previous turns. If the letter does not appear in the word, the user is charged with an incorrect guess. The user keeps guessing letters until either (1) the user has correctly guessed all the letters in the word
or (2) the user has made six incorrect guesses.

You are required to write a "guess" function that takes current word (with underscores) as input and returns a guess letter. You will use the API codes below to play 1,000 Hangman games. You have the opportunity to practice before you want to start recording your game results.

Your algorithm is permitted to use a training set of approximately 250,000 dictionary words. Your algorithm will be tested on an entirely disjoint set of 250,000 dictionary words. Please note that this means the words that you will ultimately be tested on do NOT appear in the dictionary that you are given. You are not permitted to use any dictionary other than the training dictionary we provided. This requirement will be strictly enforced by code review.

You are provided with a basic, working algorithm. This algorithm will match the provided masked string (e.g. a _ _ l e) to all possible words in the dictionary, tabulate the frequency of letters appearing in these possible words, and then guess the letter with the highest frequency of appearence that has not already been guessed. If there are no remaining words that match then it will default back to the character frequency distribution of the entire dictionary.

This benchmark strategy is successful approximately 18% of the time. Your task is to design an algorithm that significantly outperforms this benchmark.

In [26]:
import json
import requests
import random
import string
import secrets
import time
import re
import collections
import os
import numpy as np

# Import your advanced AI modules
import utils
from train import dl_model  # Updated import path
from deeplearning.models import RNN

try:
    from urllib.parse import parse_qs, urlencode, urlparse
except ImportError:
    from urlparse import parse_qs, urlparse
    from urllib import urlencode

from requests.packages.urllib3.exceptions import InsecureRequestWarning

requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

In [27]:
class HangmanAPI(object):
    def __init__(self, access_token=None, session=None, timeout=None):
        self.hangman_url = self.determine_hangman_url()
        self.access_token = access_token
        self.session = session or requests.Session()
        self.timeout = timeout
        self.guessed_letters = []
        self.misses = []  # Track wrong guesses for neural network
        
        # Load dictionary for basic fallback
        full_dictionary_location = "words_250000_train.txt"
        self.full_dictionary = self.build_dictionary(full_dictionary_location)        
        self.full_dictionary_common_letter_sorted = collections.Counter("".join(self.full_dictionary)).most_common()
        
        # Load advanced AI models
        try:
            self.n_grams = utils.build_n_gram_from_file(full_dictionary_location)
            self.dl_model = dl_model('test_one')  # Load your trained neural network
            self.use_advanced_ai = True
        except Exception:
            self.use_advanced_ai = False
        
        self.current_dictionary = []
        
    @staticmethod
    def determine_hangman_url():
        links = ['https://trexsim.com']

        data = {link: 0 for link in links}

        for link in links:

            requests.get(link)

            for i in range(10):
                s = time.time()
                requests.get(link)
                data[link] = time.time() - s

        link = sorted(data.items(), key=lambda x: x[1])[0][0]
        link += '/trexsim/hangman'
        return link

    def guess(self, word): # word input example: "_ p p _ e "
        ###############################################
        # ADVANCED AI ALGORITHM - Your trained model #
        ###############################################
        
        if self.use_advanced_ai:
            return self.guess_advanced_ai(word)
        else:
            return self.guess_basic(word)
    
    def guess_advanced_ai(self, word):
        """Advanced AI algorithm using neural networks + n-grams"""
        # clean the word so that we strip away the space characters
        # replace "_" with "." as "." indicates any character in regular expressions
        clean_word = word[::2].replace("_",".")
        len_word = len(clean_word)
        len_right_letters = len(clean_word) - clean_word.count('.')
    
        # Rule-based initial guesses for short words
        if len_right_letters == 0 and len_word in utils.LETTER_ORDER_DICT:
            order = utils.LETTER_ORDER_DICT[len_word]
            for letter in order:
                if letter not in self.guessed_letters:
                    return letter
                
        try:
            # N-gram statistical model
            ngram_probs = utils.get_n_gram_prob(self.n_grams, clean_word, self.guessed_letters)
            
            # Deep Learning neural network prediction
            best_chars, nn_probs = self.dl_model.predict(clean_word, self.misses)

            # Filter out already guessed letters and letters in the word
            nn_probs = [p if chr(i+97) not in self.misses and chr(i+97) not in clean_word else 0.0 for i,p in enumerate(nn_probs)]
            nn_probs = [p/sum(nn_probs) if sum(nn_probs) > 0 else 0 for p in nn_probs]

            # Ensemble: Combine neural network + n-gram predictions
            final_probs = np.array(nn_probs) + np.array(ngram_probs)
            best_char = chr(final_probs.argmax() + 97)
            
            # Ensure we don't repeat guesses
            if best_char not in self.guessed_letters:
                return best_char
            
        except Exception:
            pass
        
        # Fallback to basic method if AI fails
        return self.guess_basic(word)
    
    def guess_basic(self, word):
        """Original basic algorithm as fallback"""
        # clean the word so that we strip away the space characters
        # replace "_" with "." as "." indicates any character in regular expressions
        clean_word = word[::2].replace("_",".")
        
        # find length of passed word
        len_word = len(clean_word)
        
        # grab current dictionary of possible words from self object, initialize new possible words dictionary to empty
        current_dictionary = self.current_dictionary
        new_dictionary = []
        
        # iterate through all of the words in the old plausible dictionary
        for dict_word in current_dictionary:
            # continue if the word is not of the appropriate length
            if len(dict_word) != len_word:
                continue
                
            # if dictionary word is a possible match then add it to the current dictionary
            if re.match(clean_word,dict_word):
                new_dictionary.append(dict_word)
        
        # overwrite old possible words dictionary with updated version
        self.current_dictionary = new_dictionary
        
        
        # count occurrence of all characters in possible word matches
        full_dict_string = "".join(new_dictionary)
        
        c = collections.Counter(full_dict_string)
        sorted_letter_count = c.most_common()                   
        
        guess_letter = '!'
        
        # return most frequently occurring letter in all possible words that hasn't been guessed yet
        for letter,instance_count in sorted_letter_count:
            if letter not in self.guessed_letters:
                guess_letter = letter
                break
            
        # if no word matches in training dictionary, default back to ordering of full dictionary
        if guess_letter == '!':
            sorted_letter_count = self.full_dictionary_common_letter_sorted
            for letter,instance_count in sorted_letter_count:
                if letter not in self.guessed_letters:
                    guess_letter = letter
                    break            
        
        return guess_letter

    ##########################################################
    # You'll likely not need to modify any of the code below #
    ##########################################################
    
    def build_dictionary(self, dictionary_file_location):
        text_file = open(dictionary_file_location,"r")
        full_dictionary = text_file.read().splitlines()
        text_file.close()
        return full_dictionary
                
    def start_game(self, practice=True, verbose=True):
        # reset guessed letters to empty set and current plausible dictionary to the full dictionary
        self.guessed_letters = []
        self.misses = []  # Reset misses for neural network
        self.current_dictionary = self.full_dictionary
                         
        response = self.request("/new_game", {"practice":practice})
        if response.get('status')=="approved":
            game_id = response.get('game_id')
            word = response.get('word')
            tries_remains = response.get('tries_remains')
            if verbose:
                print("Successfully start a new game! Game ID: {0}. # of tries remaining: {1}. Word: {2}.".format(game_id, tries_remains, word))
            while tries_remains>0:
                # get guessed letter from user code
                guess_letter = self.guess(word)
                    
                # append guessed letter to guessed letters field in hangman object
                self.guessed_letters.append(guess_letter)
                if verbose:
                    print("Guessing letter: {0}".format(guess_letter))
                    
                try:    
                    res = self.request("/guess_letter", {"request":"guess_letter", "game_id":game_id, "letter":guess_letter})
                except HangmanAPIError:
                    print('HangmanAPIError exception caught on request.')
                    continue
                except Exception as e:
                    print('Other exception caught on request.')
                    raise e
               
                if verbose:
                    print("Sever response: {0}".format(res))
                status = res.get('status')
                tries_remains = res.get('tries_remains')
                
                # Track wrong guesses for neural network
                if 'word' in res:
                    new_word = res.get('word')
                    if new_word == word:  # Word didn't change, so it was a wrong guess
                        self.misses.append(guess_letter)
                    word = new_word
                
                if status=="success":
                    if verbose:
                        print("Successfully finished game: {0}".format(game_id))
                    return True
                elif status=="failed":
                    reason = res.get('reason', '# of tries exceeded!')
                    if verbose:
                        print("Failed game: {0}. Because of: {1}".format(game_id, reason))
                    return False
                elif status=="ongoing":
                    word = res.get('word')
        else:
            if verbose:
                print("Failed to start a new game")
        return status=="success"
        
    def my_status(self):
        return self.request("/my_status", {})
    
    def request(
            self, path, args=None, post_args=None, method=None):
        if args is None:
            args = dict()
        if post_args is not None:
            method = "POST"

        # Add `access_token` to post_args or args if it has not already been
        # included.
        if self.access_token:
            # If post_args exists, we assume that args either does not exists
            # or it does not need `access_token`.
            if post_args and "access_token" not in post_args:
                post_args["access_token"] = self.access_token
            elif "access_token" not in args:
                args["access_token"] = self.access_token

        time.sleep(0.2)

        num_retry, time_sleep = 50, 2
        for it in range(num_retry):
            try:
                response = self.session.request(
                    method or "GET",
                    self.hangman_url + path,
                    timeout=self.timeout,
                    params=args,
                    data=post_args,
                    verify=False
                )
                break
            except requests.HTTPError as e:
                response = json.loads(e.read())
                raise HangmanAPIError(response)
            except requests.exceptions.SSLError as e:
                if it + 1 == num_retry:
                    raise
                time.sleep(time_sleep)

        headers = response.headers
        if 'json' in headers['content-type']:
            result = response.json()
        elif "access_token" in parse_qs(response.text):
            query_str = parse_qs(response.text)
            if "access_token" in query_str:
                result = {"access_token": query_str["access_token"][0]}
                if "expires" in query_str:
                    result["expires"] = query_str["expires"][0]
            else:
                raise HangmanAPIError(response.json())
        else:
            raise HangmanAPIError('Maintype was not text, or querystring')

        if result and isinstance(result, dict) and result.get("error"):
            raise HangmanAPIError(result)
        return result
    
class HangmanAPIError(Exception):
    def __init__(self, result):
        self.result = result
        self.code = None
        try:
            self.type = result["error_code"]
        except (KeyError, TypeError):
            self.type = ""

        try:
            self.message = result["error_description"]
        except (KeyError, TypeError):
            try:
                self.message = result["error"]["message"]
                self.code = result["error"].get("code")
                if not self.type:
                    self.type = result["error"].get("type", "")
            except (KeyError, TypeError):
                try:
                    self.message = result["error_msg"]
                except (KeyError, TypeError):
                    self.message = result

        Exception.__init__(self, self.message)

# API Usage Examples

## To start a new game:
1. Make sure you have implemented your own "guess" method.
2. Use the access_token that we sent you to create your HangmanAPI object. 
3. Start a game by calling "start_game" method.
4. If you wish to test your function without being recorded, set "practice" parameter to 1.
5. Note: You have a rate limit of 20 new games per minute. DO NOT start more than 20 new games within one minute.

In [28]:
# Initialize with your access token
api = HangmanAPI(access_token="de81767efd117dfd071869b7d1978c", timeout=2000)


Architecture: GRU_4_1024_26
models/GRU_4_1024_26/best_GRU_4_1024.pth
models/GRU_4_1024_26/best_GRU_4_1024.pth
Loaded pretrained model from: models/GRU_4_1024_26/best_GRU_4_1024.pth
Loaded pretrained model from: models/GRU_4_1024_26/best_GRU_4_1024.pth


## Account Status Check:
**If you get "Your account has been deactivated" error, this means:**
1. You have already completed your 1000 official games submission
2. This is the expected behavior - accounts are deactivated after submission
3. Your results have been recorded by Trexquant
4. You cannot run more games, but you can check your final results below

**If you haven't submitted yet**, you can still run practice games to test your algorithm.

In [29]:
# Check account status and final results
try:
    [total_practice_runs,total_recorded_runs,total_recorded_successes,total_practice_successes] = api.my_status()
    
    print('=== FINAL SUBMISSION RESULTS ===')
    print('Practice games: %d' % total_practice_runs)
    print('Practice wins: %d' % total_practice_successes)
    if total_practice_runs > 0:
        practice_rate = total_practice_successes / total_practice_runs
        print('Practice success rate: %.3f' % practice_rate)
    
    print('\nOfficial games: %d' % total_recorded_runs)
    print('Official wins: %d' % total_recorded_successes)
    if total_recorded_runs > 0:
        official_rate = total_recorded_successes / total_recorded_runs
        print('FINAL SUCCESS RATE: %.3f' % official_rate)
        print('FINAL SUCCESS RATE: %.1f%%' % (official_rate * 100))
    
    if total_recorded_runs >= 1000:
        print('\n*** SUBMISSION COMPLETE ***')
        print('Your account has been deactivated as expected after completing 1000 games.')
    
except HangmanAPIError as e:
    if 'deactivated' in str(e):
        print('Account deactivated - this means your submission was completed successfully.')
        print('This is the expected behavior after finishing 1000 official games.')
    else:
        print('API Error:', e)
except Exception as e:
    print('Error:', e)

=== FINAL SUBMISSION RESULTS ===
Practice games: 2889
Practice wins: 547
Practice success rate: 0.189

Official games: 1000
Official wins: 548
FINAL SUCCESS RATE: 0.548
FINAL SUCCESS RATE: 54.8%

*** SUBMISSION COMPLETE ***
Your account has been deactivated as expected after completing 1000 games.


In [31]:
# Run 10 practice games to test if practice mode still works
print('Attempting to run 10 practice games...')
practice_wins = 0
practice_games = 10

for i in range(practice_games):
    try:
        print('Practice game %d/%d' % (i+1, practice_games))
        result = api.start_game(practice=1, verbose=False)
        if result:
            practice_wins += 1
            print('  - Won')
        else:
            print('  - Lost')
        
        # Rate limiting
        time.sleep(1)
        
    except HangmanAPIError as e:
        if 'deactivated' in str(e):
            print('  - Account deactivated, cannot play more games')
            break
        else:
            print('  - API Error:', e)
            break
    except Exception as e:
        print('  - Error:', e)
        break

if practice_wins > 0 or i > 0:
    games_played = i + 1 if practice_wins > 0 or not 'deactivated' in str(e) else i
    if games_played > 0:
        rate = practice_wins / games_played
        print('\nPractice session results:')
        print('Games played: %d' % games_played)
        print('Wins: %d' % practice_wins) 
        print('Success rate: %.3f' % rate)

Attempting to run 10 practice games...
Practice game 1/10
  - Account deactivated, cannot play more games


## Account Status: DEACTIVATED ✓

**Your account has been deactivated after completing your official submission.**

This means:
- You successfully completed your 1000 official games
- Your results have been recorded by Trexquant  
- Your submission is complete and final
- No additional games (practice or official) can be played
-This is permanent and cannot be reversed

**What happens next:**
1. Trexquant will evaluate all submissions
2. Results will be compared against other participants
3. You should receive feedback on your performance
4. Your advanced AI algorithm has completed its evaluation



## Playing recorded games:
Please finalize your code prior to running the cell below. Once this code executes once successfully your submission will be finalized. Our system will not allow you to rerun any additional games.

Please note that it is expected that after you successfully run this block of code that subsequent runs will result in the error message "Your account has been deactivated".

Once you've run this section of the code your submission is complete. Please send us your source code via email.

In [30]:
for i in range(1000):
    print('Playing ', i, ' th game')
    # Uncomment the following line to execute your final runs. Do not do this until you are satisfied with your submission
    api.start_game(practice=0,verbose=False)
    
    # DO NOT REMOVE as otherwise the server may lock you out for too high frequency of requests
    time.sleep(0.5)

Playing  0  th game


HangmanAPIError: {'error': 'Your account has been deactivated!'}

## To check your game statistics
1. Simply use "my_status" method.
2. Returns your total number of games, and number of wins.

In [ ]:
[total_practice_runs,total_recorded_runs,total_recorded_successes,total_practice_successes] = api.my_status() # Get my game stats: (# of tries, # of wins)
success_rate = total_recorded_successes/total_recorded_runs
print('overall success rate = %.3f' % success_rate)

overall success rate = 0.548
